# Plan: CNN + SVM Hybrid Model for Bee Health Classification

This notebook will guide you through building a simple, explainable hybrid model that combines a Convolutional Neural Network (CNN) for feature extraction with a Support Vector Machine (SVM) for classification. The goal is to categorize bees as healthy or unhealthy using both image and tabular data.

## Step-by-Step Plan

1. **Import Required Libraries**
   - Import necessary Python libraries (NumPy, pandas, scikit-learn, TensorFlow/Keras, matplotlib, etc.).

2. **Load and Explore Data**
   - Load `bee_data.csv` for labels and metadata.
   - Preview the data and check for missing values.
   - Visualize class distribution (healthy vs unhealthy).

3. **Load and Preprocess Images**
   - Load images from `data/bee_imgs/`.
   - Resize and normalize images for CNN input.
   - Match images to labels from the CSV file.

4. **Split Data**
   - Split the dataset into training and test sets (stratified by label).

5. **Build and Train CNN (Feature Extractor)**
   - Define a simple CNN architecture (or use a pre-trained model for feature extraction).
   - Train the CNN on the training data (optionally, only for a few epochs for simplicity).
   - Remove the final classification layer to use the CNN as a feature extractor.

6. **Extract Features Using CNN**
   - Pass all images through the trained CNN (up to the last pooling or flatten layer) to obtain feature vectors.

7. **Train SVM Classifier**
   - Use the extracted CNN features as input to an SVM classifier.
   - Train the SVM on the training set features and labels.

8. **Evaluate the Hybrid Model**
   - Predict on the test set using the SVM.
   - Calculate accuracy, precision, recall, F1-score, confusion matrix, and classification report.
   - Compare results with previous models (CNN-only, SVM-only).

9. **Conclusion**
   - Summarize findings and discuss the advantages and limitations of the hybrid approach.

---

**Tip:** Keep each step simple and well-commented. Use clear variable names and add explanations in markdown cells. To make the project easy to follow and present.


Copy:
Lets move on to step 5 and 6. Dont forget to import required libraries at cell 2.  Keep each step simple and well-commented. Use clear variable names and add explanations in markdown cells. To make the project easy to follow and present. You can always refer back to cnn.ipynb and svm.ipynb if needed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_recall_fscore_support
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [ ]:
bee_data = pd.read_csv("../data/bee_data.csv")

print(bee_data.head()) # Print top 5 lines
print(bee_data.shape) # Print "shape" features, # of rows & # of columns

## Step 3: Load and Preprocess Images

We will define a small helper function to load images, resize them to a fixed size, and normalize pixel values. We also connect each image to its label from the CSV.

In [ ]:
# Step 3: Load and Preprocess Images

# 1) Identify columns (adjust if your CSV uses different names)
image_col = bee_data.columns[0]  # usually the filename column is first
label_col = "health"            # in cnn.ipynb the label column is 'health'

# 2) Define image preprocessing
IMG_SIZE = 128

def preprocess_image(filename):
    img_path = os.path.join("..", "data", "bee_imgs", filename)
    img = Image.open(img_path)
    img = img.convert("RGB")
    img = img.resize((IMG_SIZE, IMG_SIZE))
    img_array = np.array(img)
    img_normalized = img_array / 255.0
    return img_normalized

# 3) Keep only filename + label (simple and clear)
image_labels_df = bee_data[[image_col, label_col]].copy()
print("Using image column:", image_col)
print("Using label column:", label_col)
print(image_labels_df.head())

## Step 4: Split Data

We will split the data into training and test sets using a stratified split, so the class distribution stays similar in both sets.

In [ ]:
# Step 4: Split Data

# Split filenames and labels (stratified to keep class balance)
x = image_labels_df[[image_col]]
y = image_labels_df[label_col]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
 )

print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_test shape: {x_test.shape}")
print(f"y_test shape: {y_test.shape}")

# Optional: show class distribution after split
print("\nTraining set class distribution:")
print(y_train.value_counts().sort_index())
print("\nTest set class distribution:")
print(y_test.value_counts().sort_index())

## Step 5: Build and Train CNN (Feature Extractor)

We will build a small CNN and train it briefly. Later we will remove the final classification layer and use the earlier layer outputs as feature vectors for SVM.

In [ ]:
# Step 5: Build and Train CNN (Feature Extractor)

# Preprocess images for CNN
x_train_processed = [preprocess_image(fname) for fname in x_train[image_col]]
x_test_processed = [preprocess_image(fname) for fname in x_test[image_col]]

x_train_processed = np.array(x_train_processed)
x_test_processed = np.array(x_test_processed)

# Encode labels to integers, then one-hot for CNN
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

num_classes = len(label_encoder.classes_)
y_train_categorical = to_categorical(y_train_encoded, num_classes)
y_test_categorical = to_categorical(y_test_encoded, num_classes)

print("x_train_processed:", x_train_processed.shape)
print("x_test_processed:", x_test_processed.shape)
print("y_train_categorical:", y_train_categorical.shape)
print("y_test_categorical:", y_test_categorical.shape)

# MODEL 1: Basic CNN (same as cnn.ipynb)
cnn_model = Sequential([
    Conv2D(16, (3, 3), activation="relu", input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D((2, 2)),
    Flatten(name="feature_layer"),
    Dense(num_classes, activation="softmax")
], name="basic_cnn")

cnn_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("=" * 50)
print("MODEL 1: Basic CNN")
print("=" * 50)
cnn_model.summary()

# Train briefly (small epoch count for simplicity)
history = cnn_model.fit(
    x_train_processed, y_train_categorical,
    validation_data=(x_test_processed, y_test_categorical),
    epochs=5,
    batch_size=32
 )

## Step 6: Extract Features Using CNN

We will remove the final classification layer and use the CNN up to the flatten layer as a feature extractor. These feature vectors will be the input to the SVM in the next step.

In [ ]:
# Step 6: Extract Features Using CNN

# Create a feature extractor model (output from the flatten layer)
feature_extractor = keras.Model(
    inputs=cnn_model.input,
    outputs=cnn_model.get_layer("feature_layer").output
)

# Extract feature vectors for train and test sets
x_train_features = feature_extractor.predict(x_train_processed, batch_size=32)
x_test_features = feature_extractor.predict(x_test_processed, batch_size=32)

print("x_train_features:", x_train_features.shape)
print("x_test_features:", x_test_features.shape)

## Step 7: Train SVM Classifier

We will train an SVM using the CNN feature vectors as input. This keeps the classifier simple and effective.

In [ ]:
# Step 7: Train SVM Classifier

# Use CNN features as input to SVM
svm_model = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", C=1.0, gamma="scale")
)

svm_model.fit(x_train_features, y_train)
print("SVM training complete.")

## Step 8: Evaluate the Hybrid Model

We will evaluate the SVM using the CNN features and report accuracy, precision, recall, F1-score, and a confusion matrix.

In [ ]:
# Step 8: Evaluate the Hybrid Model

# Predict on the test set
y_pred = svm_model.predict(x_test_features)

# Metrics
acc = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="weighted"
 )

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}\n")

# Confusion matrix and classification report
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Step 9: Conclusion

### Summary of Findings
- The hybrid approach uses the CNN as a feature extractor and the SVM as the final classifier.
- This usually performs better than using only raw pixels with SVM, and can sometimes be more stable than a full CNN on small datasets.
- The evaluation metrics (accuracy, precision, recall, F1-score) show how well the model separates healthy vs unhealthy bees.

### Advantages of the CNN + SVM Hybrid
- **Better features:** CNN learns meaningful visual patterns (e.g., shape, texture).
- **Simpler classifier:** SVM can work well on smaller datasets with good features.
- **Interpretability:** Feature extraction and classification are separated, making the pipeline easier to explain.
- **Flexibility:** You can swap the CNN or SVM without redesigning the full pipeline.

### Limitations and Things to Watch
- **Data imbalance:** If healthy bees dominate, the model may bias toward “healthy.”
- **Small dataset risk:** With few images, the CNN features may not generalize well.
- **Two-stage training:** You must train the CNN first, then the SVM (extra steps).
- **Feature quality matters:** A very small CNN may produce weak features.
- **Compute cost:** Feature extraction for all images can be slow if the dataset grows.

### Ideas for Improvement (Simple and Explainable)
- Use class weights or balanced sampling to reduce bias.
- Try a slightly larger CNN or a pre-trained backbone (transfer learning).
- Add simple image augmentation (flip, rotate) to improve generalization.
- Test PCA only if feature size becomes too large or training becomes slow.